# CSE476 – E-Commerce Order Fulfilment Orchestrator (Live Demo Notebook)

**Academic Task:** CA2 – Task 2 (Group 11 – Individual Project)  
**Framework:** Microsoft Agent Framework (`agent-framework-ollama`) + Local Ollama (`qwen3:4b`)  
**Architecture:** 5-Agent Sequential Pipeline with Fraud Escalation Route, Shared `OrderState`, Persistent SQLite Customer Memory, and Append-Only JSONL Execution Log.

This notebook executes live against the project backend (using an isolated notebook SQLite DB copy so it can be re-run cleanly top-to-bottom) and displays real multi-agent execution traces across 5 scenarios.

## 1. Library Imports & Path Setup

In [1]:
import html
import json
import sys
from pathlib import Path
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.orchestrator import AGENT_ROSTER, OrderFulfilmentOrchestrator
from agents.validation_agent import build_validation_agent
from agents.inventory_agent import build_inventory_agent
from agents.fraud_agent import build_fraud_agent
from agents.pricing_agent import build_pricing_agent
from agents.dispatch_agent import build_dispatch_agent

def render_table(rows, columns=None):
    """Render a list of dicts as a clean styled HTML table (zero OpenBLAS overhead)."""
    if not rows:
        print('(no rows)')
        return
    cols = columns or list(rows[0].keys())
    th = ''.join(f'<th style="text-align:left;padding:6px 10px;background:#0F172A;color:#FFF;">{html.escape(str(c))}</th>' for c in cols)
    body_rows = []
    for idx, row in enumerate(rows):
        bg = '#F8FAFC' if idx % 2 else '#FFFFFF'
        tds = ''.join(f'<td style="padding:6px 10px;border-bottom:1px solid #E2E8F0;">{html.escape(str(row.get(c, "")))}</td>' for c in cols)
        body_rows.append(f'<tr style="background:{bg};">{tds}</tr>')
    table_html = f'<table style="border-collapse:collapse;width:100%;font-size:13px;"><thead><tr>{th}</tr></thead><tbody>{"".join(body_rows)}</tbody></table>'
    display(HTML(table_html))

print('Project Root:', PROJECT_ROOT)

Project Root: D:\AI\AgentProjects\Ecommerce_Order_Orchestrator


## 2. Dataset Loading & SQLite Database Initialization

In [2]:
products_raw = json.loads((PROJECT_ROOT / 'data' / 'products.json').read_text(encoding='utf-8'))
print('Catalog metadata:', products_raw['metadata']['description'])
render_table(products_raw['products'], ['product_id', 'name', 'price', 'stock', 'category', 'substitute_id'])

# Initialize orchestrator with a dedicated notebook DB and append-only JSONL log
orch = OrderFulfilmentOrchestrator(
    db_path=PROJECT_ROOT / 'memory' / 'notebook_demo.db',
    products_json_path=PROJECT_ROOT / 'data' / 'products.json',
    log_path=PROJECT_ROOT / 'logs' / 'execution.jsonl',
)
orch.db.reset_demo_data()

print('\nSeeded Sample Customers (with prior-order memory in SQLite):')
cust_rows = []
for c in orch.db.list_customers():
    h = orch.db.get_customer_history(c['customer_id'])
    cust_rows.append({
        'Customer ID': c['customer_id'],
        'Name': c['name'],
        'Prior Orders': h['total_prior_orders'],
        'Completed': h['completed_orders'],
        'Failed Payments': h['failed_payments'],
        'Cancelled': h['cancelled_orders'],
        'Prior Fraud Flags': h['prior_fraud_flags'],
    })
render_table(cust_rows)

Catalog metadata: Sample demonstration product catalog for CSE476 E-Commerce Order Fulfilment Orchestrator (not real store data).


product_id,name,price,stock,category,substitute_id
P101,Wireless Headphones,2000.0,15,Audio,P103
P102,Smart Watch,3500.0,8,Wearables,P101
P103,Bluetooth Speaker,1500.0,3,Audio,P101
P104,Power Bank,1000.0,0,Accessories,P105
P105,Keyboard,1200.0,12,Accessories,P103



Seeded Sample Customers (with prior-order memory in SQLite):


Customer ID,Name,Prior Orders,Completed,Failed Payments,Cancelled,Prior Fraud Flags
C101,Aarav Sharma,1,1,0,0,0
C102,Priya Nair,4,4,0,0,0
C103,Rohan Verma,3,0,0,3,0
C104,Vikram Mehta,3,0,3,0,1
C105,Ananya Iyer,0,0,0,0,0


## 3. Microsoft Agent Framework & Ollama Initialization

In [3]:
ollama_ok, health_msg = orch.ollama_status()
print('Ollama Status:', health_msg)

# Instantiate all 5 real Microsoft Agent Framework agents and inspect registered tools
client = orch._create_ollama_client()
framework_agents = [
    build_validation_agent(client, orch.toolkit),
    build_inventory_agent(client, orch.toolkit),
    build_fraud_agent(client, orch.toolkit),
    build_pricing_agent(client, orch.toolkit),
    build_dispatch_agent(client, orch.toolkit),
]

agent_summary = []
for meta, ag in zip(AGENT_ROSTER, framework_agents):
    agent_summary.append({
        'Stage': meta['stage'],
        'Agent Name': ag.name,
        'Framework Class': type(ag).__name__,
        'Module': meta['module'],
        'Tools': ', '.join(meta['tools']),
    })
render_table(agent_summary)

Ollama Status: Ollama online at http://localhost:11434 with model 'qwen3:4b'.


Stage,Agent Name,Framework Class,Module,Tools
1,Validation Agent,Agent,agents/validation_agent.py,"check_product_exists, validate_address, verify_payment"
2,Inventory Agent,Agent,agents/inventory_agent.py,"get_stock_level, reserve_stock, suggest_substitute"
3,Fraud Check Agent,Agent,agents/fraud_agent.py,"get_customer_history, calculate_fraud_score, record_fraud_flag"
4,Pricing Agent,Agent,agents/pricing_agent.py,"get_product_price, calculate_discount, calculate_final_total"
5,Dispatch Coordinator,Agent,agents/dispatch_agent.py,"select_shipping_method, generate_tracking_id, confirm_dispatch"


## 4. Helper to Print Multi-Agent Outputs & Execution Trace

In [4]:
def display_order_outcome(title, state):
    print('=' * 85)
    print(f'SCENARIO: {title}')
    print(f'Order ID: {state.order_id} | Customer: {state.customer_id} ({state.customer_name}) | Mode: {state.execution_mode}')
    print(f'FINAL DECISION: {state.decision}')
    print(
        f'Stages -> Validation: {state.validation_status} | Inventory: {state.inventory_status} | '
        f'Fraud: {state.fraud_status} ({state.fraud_score}/100) | Pricing: {state.pricing_status} | '
        f'Dispatch: {state.dispatch_status}'
    )
    if state.decision == 'DISPATCHED':
        print(
            f'Pricing -> Subtotal: INR {state.subtotal:,.2f} | Discount: INR {state.discount:,.2f} | '
            f'Tax (18%): INR {state.tax:,.2f} | Final Total: INR {state.final_total:,.2f}'
        )
        print(f'Dispatch -> {state.shipping_method} | Tracking ID: {state.tracking_id}')
    elif state.hold_reason:
        print(f'Reason -> {state.hold_reason}')

    print('\nIndividual Agent Outputs:')
    for agent_name, text in state.agent_summaries.items():
        print(f'  - [{agent_name}] {text}')

    print('\nMulti-Agent Execution Trace:')
    render_table(
        state.execution_trace,
        ['timestamp', 'agent_name', 'tool_used', 'action', 'result', 'execution_mode', 'explanation'],
    )

## 5. Scenario 1 — Live Multi-Agent Order Fulfilment (Trusted Repeat Customer -> DISPATCHED)
Runs end-to-end through **all 5 Microsoft Agent Framework agents** using live local Ollama (`qwen3:4b`) when online (or transparent DEMO fallback if offline).

In [5]:
run_mode = 'OLLAMA' if ollama_ok else 'DEMO'
state_success = await orch.process_order_async(
    customer_id='C102',
    customer_name='Priya Nair',
    product_id='P102',  # Smart Watch - INR 3,500 (qualifies for 5% sample discount)
    quantity=1,
    address='18 Indiranagar 100ft Rd, Bengaluru, Karnataka 560038',
    payment_status='PAID',
    mode=run_mode,
)
display_order_outcome('Scenario 1: Successful Trusted Repeat Customer Order', state_success)

SCENARIO: Scenario 1: Successful Trusted Repeat Customer Order
Order ID: ORD-0CA5B1DD | Customer: C102 (Priya Nair) | Mode: OLLAMA
FINAL DECISION: DISPATCHED
Stages -> Validation: VALID | Inventory: RESERVED | Fraud: LOW (0/100) | Pricing: PRICED | Dispatch: DISPATCHED
Pricing -> Subtotal: INR 3,500.00 | Discount: INR 175.00 | Tax (18%): INR 598.50 | Final Total: INR 3,923.50
Dispatch -> Express Metro Courier (1-2 Business Days) | Tracking ID: TRK-643E8132

Individual Agent Outputs:
  - [Validation Agent] Okay, let's tackle this validation step by step. The user wants me to validate the order ORD-0CA5B1DD using the provided product_id, address, and payment_status. I need to call the three tools in order: check_product_exists, validate_address, and verify_payment.

First, I'll check the product exists. The function check_product_exists requires order_id and product_id. The order_id here is ORD-0CA5B1DD and product_id is P1: Wait, no, the user wrote product_id: P102. So product_id is P10

timestamp,agent_name,tool_used,action,result,execution_mode,explanation
2026-11-18T12:29:34Z,Orchestrator,pipeline_guard,workflow_started,OLLAMA,OLLAMA,Started 5-agent fulfilment workflow in OLLAMA mode for ORD-0CA5B1DD.
2026-11-18T12:30:02Z,Validation Agent,check_product_exists,check_product,VALID,OLLAMA,Product P102 (Smart Watch) verified at INR 3500.00 x 1.
2026-11-18T12:30:02Z,Validation Agent,validate_address,validate_address,VALID,OLLAMA,"Delivery address verified: 18 Indiranagar 100ft Rd, Bengaluru, Karnataka 560038"
2026-11-18T12:30:02Z,Validation Agent,verify_payment,verify_payment,VALID,OLLAMA,Payment status confirmed as PAID.
2026-11-18T12:30:13Z,Inventory Agent,get_stock_level,check_stock,AVAILABLE,OLLAMA,Smart Watch (P102) has 8 unit(s) in stock; requested 1.
2026-11-18T12:30:34Z,Inventory Agent,reserve_stock,reserve_stock,RESERVED,OLLAMA,Reserved 1 unit(s) of P102. Remaining stock: 7.
2026-11-18T12:30:55Z,Fraud Check Agent,get_customer_history,retrieve_memory,RETRIEVED,OLLAMA,"Retrieved history for C102 (Priya Nair): 4 prior order(s), 0 failed payment(s), 0 cancellation(s), 0 prior fraud flag(s)."
2026-11-18T12:30:55Z,Fraud Check Agent,calculate_fraud_score,score_fraud_risk,LOW,OLLAMA,Fraud score = 0/100 (LOW). No risk rules triggered.
2026-11-18T12:31:15Z,Pricing Agent,get_product_price,get_price,OK,OLLAMA,Unit price for P102 (Smart Watch) = INR 3500.00 x 1 -> Subtotal INR 3500.00.
2026-11-18T12:31:15Z,Pricing Agent,calculate_discount,calculate_discount,OK,OLLAMA,Discount calculated: INR 175.00. 5% demo discount applied (subtotal >= INR 3000.00).


## 6. Scenario 2 — Fraud Escalation Route (High-Risk Customer `C104` -> HELD_FOR_REVIEW)
Demonstrates persistent SQLite history lookup (`C104` has 3 prior failed payments and 1 prior fraud flag -> score ≥ 60). The pipeline halts after the Fraud Check Agent and never reaches Dispatch.

In [6]:
state_fraud = await orch.process_order_async(
    customer_id='C104',
    customer_name='Vikram Mehta',
    product_id='P101',
    quantity=1,
    address='901 Linking Road, Mumbai, Maharashtra 400050',
    payment_status='PAID',
    mode='DEMO',  # Switch to 'OLLAMA' to invoke live qwen3:4b on this scenario as well
)
display_order_outcome('Scenario 2: High Fraud Risk Escalation (Held for Review)', state_fraud)

SCENARIO: Scenario 2: High Fraud Risk Escalation (Held for Review)
Order ID: ORD-91EF25EB | Customer: C104 (Vikram Mehta) | Mode: DEMO
FINAL DECISION: HELD_FOR_REVIEW
Stages -> Validation: VALID | Inventory: RESERVED | Fraud: HIGH (60/100) | Pricing: SKIPPED | Dispatch: HELD
Reason -> Fraud escalation triggered (Risk: HIGH, Score: 60/100). Multiple prior failed payments (3): +30; Prior fraud flag(s) on record (1): +30 Reserved stock held pending manual review.

Individual Agent Outputs:
  - [Validation Agent] VALID: Product P101, delivery address, and PAID status verified.
  - [Inventory Agent] RESERVED: Allocated 1 unit(s) of P101.
  - [Fraud Check Agent] HIGH RISK (Score 60/100): Multiple prior failed payments (3): +30; Prior fraud flag(s) on record (1): +30

Multi-Agent Execution Trace:


timestamp,agent_name,tool_used,action,result,execution_mode,explanation
2026-11-18T12:31:35Z,Orchestrator,pipeline_guard,workflow_started,DEMO,DEMO,Started 5-agent fulfilment workflow in DEMO mode for ORD-91EF25EB.
2026-11-18T12:31:35Z,Validation Agent,check_product_exists,check_product,VALID,DEMO,Product P101 (Wireless Headphones) verified at INR 2000.00 x 1.
2026-11-18T12:31:35Z,Validation Agent,validate_address,validate_address,VALID,DEMO,"Delivery address verified: 901 Linking Road, Mumbai, Maharashtra 400050"
2026-11-18T12:31:35Z,Validation Agent,verify_payment,verify_payment,VALID,DEMO,Payment status confirmed as PAID.
2026-11-18T12:31:35Z,Inventory Agent,get_stock_level,check_stock,AVAILABLE,DEMO,Wireless Headphones (P101) has 15 unit(s) in stock; requested 1.
2026-11-18T12:31:35Z,Inventory Agent,reserve_stock,reserve_stock,RESERVED,DEMO,Reserved 1 unit(s) of P101. Remaining stock: 14.
2026-11-18T12:31:35Z,Fraud Check Agent,get_customer_history,retrieve_memory,RETRIEVED,DEMO,"Retrieved history for C104 (Vikram Mehta): 3 prior order(s), 3 failed payment(s), 0 cancellation(s), 1 prior fraud flag(s)."
2026-11-18T12:31:35Z,Fraud Check Agent,calculate_fraud_score,score_fraud_risk,HIGH,DEMO,Fraud score = 60/100 (HIGH). Multiple prior failed payments (3): +30; Prior fraud flag(s) on record (1): +30
2026-11-18T12:31:35Z,Fraud Check Agent,record_fraud_flag,record_fraud_flag,FLAGGED,DEMO,Fraud flag #5 recorded in SQLite for C104 (HIGH): Multiple prior failed payments (3): +30; Prior fraud flag(s) on record (1): +30
2026-11-18T12:31:35Z,Orchestrator,pipeline_guard,fraud_escalation,HELD_FOR_REVIEW,DEMO,"Fraud escalation triggered (Risk: HIGH, Score: 60/100). Multiple prior failed payments (3): +30; Prior fraud flag(s) on record (1): +30 Reserved stock held pending manual review."


## 7. Scenario 3 — Stock Shortfall & Substitute Suggestion (`P104` Out of Stock)

In [7]:
state_shortfall = await orch.process_order_async(
    customer_id='C101',
    customer_name='Aarav Sharma',
    product_id='P104',  # Power Bank (0 in stock)
    quantity=1,
    address='42 MG Road, Bengaluru, Karnataka 560001',
    payment_status='PAID',
    mode='DEMO',
)
display_order_outcome('Scenario 3: Out-of-Stock Shortfall with Substitute', state_shortfall)

SCENARIO: Scenario 3: Out-of-Stock Shortfall with Substitute
Order ID: ORD-1F77225E | Customer: C101 (Aarav Sharma) | Mode: DEMO
FINAL DECISION: SHORTFALL
Stages -> Validation: VALID | Inventory: SHORTFALL | Fraud: PENDING (0/100) | Pricing: SKIPPED | Dispatch: NOT_DISPATCHED
Reason -> Insufficient stock for P104 (qty 1). Suggested substitute: P105 (Keyboard).

Individual Agent Outputs:
  - [Validation Agent] VALID: Product P104, delivery address, and PAID status verified.
  - [Inventory Agent] SHORTFALL: Insufficient stock for P104. Suggested substitute: P105 (Keyboard - INR 1200.00).

Multi-Agent Execution Trace:


timestamp,agent_name,tool_used,action,result,execution_mode,explanation
2026-11-18T12:31:35Z,Orchestrator,pipeline_guard,workflow_started,DEMO,DEMO,Started 5-agent fulfilment workflow in DEMO mode for ORD-1F77225E.
2026-11-18T12:31:35Z,Validation Agent,check_product_exists,check_product,VALID,DEMO,Product P104 (Power Bank) verified at INR 1000.00 x 1.
2026-11-18T12:31:35Z,Validation Agent,validate_address,validate_address,VALID,DEMO,"Delivery address verified: 42 MG Road, Bengaluru, Karnataka 560001"
2026-11-18T12:31:35Z,Validation Agent,verify_payment,verify_payment,VALID,DEMO,Payment status confirmed as PAID.
2026-11-18T12:31:35Z,Inventory Agent,get_stock_level,check_stock,INSUFFICIENT,DEMO,Power Bank (P104) has 0 unit(s) in stock; requested 1.
2026-11-18T12:31:35Z,Inventory Agent,reserve_stock,reserve_stock,SHORTFALL,DEMO,"Stock shortfall for P104: requested 1, available 0. Stock unchanged."
2026-11-18T12:31:35Z,Inventory Agent,suggest_substitute,suggest_substitute,SUGGESTED,DEMO,Suggested substitute for P104: P105 (Keyboard) at INR 1200.00 (12 in stock).
2026-11-18T12:31:35Z,Orchestrator,pipeline_guard,pipeline_stop,SHORTFALL,DEMO,Stopped after Inventory Agent: Insufficient stock for P104 (qty 1). Suggested substitute: P105 (Keyboard).


## 8. Scenario 4 — Payment Failure (`UNPAID` -> REJECTED at Validation)

In [8]:
state_unpaid = await orch.process_order_async(
    customer_id='C101',
    customer_name='Aarav Sharma',
    product_id='P101',
    quantity=1,
    address='42 MG Road, Bengaluru, Karnataka 560001',
    payment_status='UNPAID',
    mode='DEMO',
)
display_order_outcome('Scenario 4: Unpaid Order Rejected at Validation', state_unpaid)

SCENARIO: Scenario 4: Unpaid Order Rejected at Validation
Order ID: ORD-19B5DD41 | Customer: C101 (Aarav Sharma) | Mode: DEMO
FINAL DECISION: REJECTED
Stages -> Validation: INVALID | Inventory: PENDING | Fraud: PENDING (0/100) | Pricing: SKIPPED | Dispatch: NOT_DISPATCHED
Reason -> Payment status is 'UNPAID'; order cannot proceed unless PAID.

Individual Agent Outputs:
  - [Validation Agent] INVALID: Payment status is 'UNPAID'; order cannot proceed unless PAID.

Multi-Agent Execution Trace:


timestamp,agent_name,tool_used,action,result,execution_mode,explanation
2026-11-18T12:31:35Z,Orchestrator,pipeline_guard,workflow_started,DEMO,DEMO,Started 5-agent fulfilment workflow in DEMO mode for ORD-19B5DD41.
2026-11-18T12:31:35Z,Validation Agent,check_product_exists,check_product,VALID,DEMO,Product P101 (Wireless Headphones) verified at INR 2000.00 x 1.
2026-11-18T12:31:35Z,Validation Agent,validate_address,validate_address,VALID,DEMO,"Delivery address verified: 42 MG Road, Bengaluru, Karnataka 560001"
2026-11-18T12:31:35Z,Validation Agent,verify_payment,verify_payment,INVALID,DEMO,Payment status is 'UNPAID'; order cannot proceed unless PAID.
2026-11-18T12:31:35Z,Orchestrator,pipeline_guard,pipeline_stop,REJECTED,DEMO,Stopped after Validation Agent: Payment status is 'UNPAID'; order cannot proceed unless PAID.


## 9. Scenario 5 — Repeat Customer Persistent Memory Across Orders (`C105`)

In [9]:
# C105 starts with 0 prior orders; placing a high-value order (>= 5,000) scores 40 (MEDIUM) and saves a flag.
state_c105_first = await orch.process_order_async(
    customer_id='C105',
    customer_name='Ananya Iyer',
    product_id='P102',
    quantity=2,  # INR 7,000
    address='14 Civil Lines, Jaipur, Rajasthan 302006',
    payment_status='PAID',
    mode='DEMO',
)
print(f'First C105 order  -> Prior Orders: {state_c105_first.customer_history_summary["total_prior_orders"]}, Fraud Score: {state_c105_first.fraud_score} ({state_c105_first.fraud_status}), Decision: {state_c105_first.decision}')

# Second order from C105 retrieves the newly persisted history & fraud flag from SQLite!
state_c105_second = await orch.process_order_async(
    customer_id='C105',
    customer_name='Ananya Iyer',
    product_id='P105',
    quantity=1,  # INR 1,200
    address='14 Civil Lines, Jaipur, Rajasthan 302006',
    payment_status='PAID',
    mode='DEMO',
)
print(f'Second C105 order -> Prior Orders: {state_c105_second.customer_history_summary["total_prior_orders"]}, Prior Flags: {state_c105_second.customer_history_summary["prior_fraud_flags"]}, Fraud Score: {state_c105_second.fraud_score} ({state_c105_second.fraud_status}), Decision: {state_c105_second.decision}')

First C105 order  -> Prior Orders: 0, Fraud Score: 40 (MEDIUM), Decision: DISPATCHED
Second C105 order -> Prior Orders: 1, Prior Flags: 1, Fraud Score: 30 (MEDIUM), Decision: DISPATCHED


## 10. Append-Only Execution Log Verification (`logs/execution.jsonl`)

In [10]:
all_logs = orch.logger.read_all()
print(f'Total append-only entries in logs/execution.jsonl: {len(all_logs)}')
render_table(all_logs[-12:], ['timestamp', 'order_id', 'agent_name', 'tool_used', 'result', 'execution_mode'])

Total append-only entries in logs/execution.jsonl: 111


timestamp,order_id,agent_name,tool_used,result,execution_mode
2026-11-18T12:31:35Z,ORD-A9457982,Validation Agent,verify_payment,VALID,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Inventory Agent,get_stock_level,AVAILABLE,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Inventory Agent,reserve_stock,RESERVED,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Fraud Check Agent,get_customer_history,RETRIEVED,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Fraud Check Agent,calculate_fraud_score,MEDIUM,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Fraud Check Agent,record_fraud_flag,FLAGGED,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Pricing Agent,get_product_price,OK,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Pricing Agent,calculate_discount,OK,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Pricing Agent,calculate_final_total,PRICED,DEMO
2026-11-18T12:31:35Z,ORD-A9457982,Dispatch Coordinator,select_shipping_method,SELECTED,DEMO
